# 02 Preprocessing
Tokenization of the texts, the choice of the word embeddings (GloVe), lowercasing, stopwords, stemming and lemmatization, punctuation and format cleaning, and the input of both models: for model 1 the average GloVe vector of each text, saved to `features/model1.npz`; for model 2 the padded token indices of each text and the GloVe embedding matrix, saved to `features/model2.npz`.

**GPU version (1.11):** the vocabulary of model 2 also holds the validation and test words that have a GloVe vector. The outputs are from the run on Colab. Run this branch in its own clone: `features/` is not in git, so a run in the folder of `main` would overwrite the files of the CPU run.

In [1]:
# setup: on Colab, connect Google Drive, where the features are saved
import os
import sys

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    os.chdir("/content/drive/MyDrive/nlp-project1-art")
    %pip install -q gensim datasets nltk

Mounted at /content/drive


## The decisions in short
The measurements and reasons are in the sections below.

| Decision | Reason and measurement |
|---|---|
| GloVe, 300 dimensions (`glove-wiki-gigaword-300`) | 99.84% of the training tokens have a vector; word2vec about 78%; fastText lacks `n't` |
| `word_tokenize` | Separates punctuation and `n't`, which then match GloVe |
| Lowercase | GloVe is uncased: coverage 87.43% to 99.84% |
| No stemming, no lemmatization | Stems lower the coverage to 93.42%; inflected forms keep tense and number |
| Keep stopwords and punctuation | The list has *not*, *no*, *before*, *after*; without stopwords, 357 pairs have the same words and counts |
| Skip unknown words, keep rare words | 0.16% of the training tokens are unknown; a frozen UNK vector cannot learn; GloVe covers rare words |
| No format cleaning | No HTML or web addresses; 38 of 67,204 training texts are non-ASCII |
| Pad to 34 (model 2) | Longest training text; only one validation example loses its final period |
| Vocabulary of model 2: every word with a GloVe vector | Model 2 skips the same words as model 1; the training input does not change |
| All four texts as features | The task needs them; what the observations add is measured in `03_model1_embeddings.ipynb` and `04_model2_rnn.ipynb` |

Load the train, validation and test splits (like on slide 9 of the course project slides).

In [2]:
from datasets import load_dataset

train = load_dataset("amarfurt/art-nlp-hs26", split="train")
valid = load_dataset("amarfurt/art-nlp-hs26", split="validation")
test = load_dataset("amarfurt/art-nlp-hs26", split="test")

print(len(train), len(valid), len(test))

README.md:   0%|          | 0.00/1.86k [00:00<?, ?B/s]

anli/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.12MB            

anli/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

anli/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  130kB            

anli/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

anli/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  200kB            

anli/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/16801 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1532 [00:00<?, ? examples/s]

16801 1000 1532


# Part 1: decisions for both models

## Tokenization
`word_tokenize` needs the NLTK resource `punkt_tab`.


In [3]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

Compare splitting at whitespace with NLTK's `word_tokenize` on one example (the first short training hypothesis that contains both a comma and a contraction).


In [4]:
text = train[269]['hypothesis_1']
print(text)
print(text.split())
print(nltk.word_tokenize(text))

He didn't listen to them, and had to go to the hospital.
['He', "didn't", 'listen', 'to', 'them,', 'and', 'had', 'to', 'go', 'to', 'the', 'hospital.']
['He', 'did', "n't", 'listen', 'to', 'them', ',', 'and', 'had', 'to', 'go', 'to', 'the', 'hospital', '.']


`split()` cuts the text only at spaces, so punctuation stays attached to words (`them,`, `hospital.`). `word_tokenize` separates punctuation (`them` and `,`) and contractions (`did` and `n't`) from the words. This way the words match the vocabulary of the word embeddings, and negations like `n't` stay their own tokens. `word_tokenize` is used for all texts.

## Word embeddings
Load the pretrained GloVe embeddings (300 dimensions) with gensim's downloader, and look at the vector size, the vocabulary size and the first words.

In [5]:
import gensim.downloader as api
wv = api.load('glove-wiki-gigaword-300')


[==================================================] 100.0% 376.1/376.1MB downloaded


In [6]:
print(f'Vector size: {wv.vector_size}')
print(f'Vocabulary size: {len(wv.index_to_key)}')
for index, word in enumerate(wv.index_to_key[:10]):
    print(f"word #{index}: {word}")

Vector size: 300
Vocabulary size: 400000
word #0: the
word #1: ,
word #2: .
word #3: of
word #4: to
word #5: and
word #6: in
word #7: a
word #8: "
word #9: 's


Count the words in the training data. Tokenize all texts (O1, O2, h1, h2) and count how many words there are in total (tokens) and how many distinct words (types), once as written and once lowercased. This gives the list of words that need a vector.

In [7]:
columns = ['observation_1', 'observation_2', 'hypothesis_1', 'hypothesis_2']

tokens = []
for column in columns:
    for text in train[column]:
        tokens.extend(nltk.word_tokenize(text))

fdist = nltk.FreqDist(tokens)
fdist_lower = nltk.FreqDist(token.lower() for token in tokens)
print('Tokens:', len(tokens))
print('Types:', len(fdist))
print('Types (lowercased):', len(fdist_lower))
print(f'Type-token-ratio: {len(fdist) / len(tokens):.2%}')

Tokens: 655481
Types: 17789
Types (lowercased): 16209
Type-token-ratio: 2.71%


Share of types and tokens that have a vector in GloVe, and the most frequent words without a vector.


In [8]:
def coverage(wv, fdist):
    known = [word for word in fdist if word in wv]
    known_tokens = sum(fdist[word] for word in known)
    print(f'Known types: {len(known) / len(fdist):.2%}')
    print(f'Known tokens: {known_tokens / fdist.N():.2%}')
    unknown = [word for word, count in fdist.most_common() if word not in wv]
    print('Most frequent unknown:', unknown[:15])

print('GloVe, cased:')
coverage(wv, fdist)
print('GloVe, lowercased:')
coverage(wv, fdist_lower)


GloVe, cased:
Known types: 73.82%
Known tokens: 87.43%
Most frequent unknown: ['I', 'The', 'He', 'She', 'My', 'It', 'A', 'They', 'Gina', 'Amy', 'Bob', 'Tim', 'Sam', 'We', 'John']
GloVe, lowercased:
Known types: 95.40%
Known tokens: 99.84%
Most frequent unknown: ['mturk', '..', 'selfie', 'zaiden', 'a+', 'antonela', 'fariah', 'myah', 'shalil', 'rubix', 'kelsea', 'jacelyn', 'lamay', 'marisse', 'lakyn']


GloVe only contains lowercase words: cased, only 87.43% of the tokens have a vector; lowercased, 99.84%, so almost every word the model sees has a vector. The remaining unknown words are typos, hyphenated compounds and rare names. **All texts are therefore lowercased.**

For comparison: the coverage of fastText and word2vec (the embeddings from the course exercise). Each is loaded, measured and deleted again to free memory.

In [9]:
for name in ['fasttext-wiki-news-subwords-300', 'word2vec-google-news-300']:
    other_wv = api.load(name)
    print(f'{name}, cased:')
    coverage(other_wv, fdist)
    print(f'{name}, lowercased:')
    coverage(other_wv, fdist_lower)
    del other_wv


[==================================================] 100.0% 958.5/958.4MB downloaded
fasttext-wiki-news-subwords-300, cased:
Known types: 96.96%
Known tokens: 99.38%
Most frequent unknown: ["n't", '``', "''", 'mTurk', 'Zaiden', 'mturk', 'A+', 'Antonela', 'Fariah', 'Mturk', 'Shalil', 'rubix', 'Marisse', 'Lakyn', 'Tanice']
fasttext-wiki-news-subwords-300, lowercased:
Known types: 93.97%
Known tokens: 99.11%
Most frequent unknown: ["n't", '``', "''", 'kelsi', 'marcy', 'rhonda', 'joanie', 'mindy', 'mturk', 'horace', 'tabitha', 'hallie', 'horatio', 'juanita', 'callie']
[==================================================] 100.0% 1662.8/1662.8MB downloaded
word2vec-google-news-300, cased:
Known types: 94.13%
Known tokens: 78.34%
Most frequent unknown: ['.', 'to', 'a', 'and', ',', 'of', "'s", '!', '``', "'", '10', '20', 'didnt', '100', 'cancelled']
word2vec-google-news-300, lowercased:
Known types: 88.88%
Known tokens: 77.62%
Most frequent unknown: ['.', 'to', 'a', 'and', ',', 'of', "'s", '!',

Conclusion: GloVe covers 99.84% of the training tokens. word2vec, used in the course exercise, covers only about 78%: it has no vectors for punctuation and frequent words like *to*, *a*, *and*, *of*. fastText covers 99.38%, but with these downloaded vectors and this lookup, the contracted-negation token `n't` (*didn't*, *wasn't*) would be skipped. So GloVe is more suitable for this project.


## Stopwords
Download NLTK's English stopword list and check which important words it contains.

In [10]:
nltk.download('stopwords')
stopwords = set(nltk.corpus.stopwords.words('english'))
print('Stopwords:', len(stopwords))
print([word for word in ['not', 'no', 'never', 'before', 'after', 'then'] if word in stopwords])

Stopwords: 198
['not', 'no', 'before', 'after', 'then']


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


How many words in the texts are stopwords? Share of the training tokens that are stopwords.

In [11]:
stopword_tokens = sum(fdist_lower[word] for word in fdist_lower if word in stopwords)
print(f'Stopword tokens: {stopword_tokens / fdist_lower.N():.2%}')

Stopword tokens: 38.95%


Number of hypothesis pairs whose differing words are all stopwords, and what removing the stopwords would do to these pairs: model 1 ignores the word order, model 2 does not.

In [12]:
only_stopwords, same_counts, same_sequence = 0, 0, 0
for h1, h2 in zip(train['hypothesis_1'], train['hypothesis_2']):
    # lists of the lowercased words: they keep the word order and repeated words
    words_1 = [word.lower() for word in nltk.word_tokenize(h1)]
    words_2 = [word.lower() for word in nltk.word_tokenize(h2)]
    difference = set(words_1) ^ set(words_2)  # words that occur in only one of the two hypotheses
    if difference and difference <= stopwords:
        only_stopwords += 1
        # the two hypotheses without their stopwords
        kept_1 = [word for word in words_1 if word not in stopwords]
        kept_2 = [word for word in words_2 if word not in stopwords]
        # same words with the same counts, in any order: model 1 gets the same average vector (True counts as 1)
        same_counts += nltk.FreqDist(kept_1) == nltk.FreqDist(kept_2)
        # same words in the same order: model 2 gets the same input
        same_sequence += kept_1 == kept_2
print('Pairs whose differing words are all stopwords:', only_stopwords)
print(f'Share: {only_stopwords / len(train):.2%}')
print('Without stopwords, same words and counts (model 1):', same_counts)
print('Without stopwords, same word sequence (model 2):', same_sequence)

Pairs whose differing words are all stopwords: 367
Share: 2.18%
Without stopwords, same words and counts (model 1): 357
Without stopwords, same word sequence (model 2): 263


NLTK's list contains *not*, *no*, *before*, *after* and *then*, which change the meaning of a sentence. Stopwords are 38.95% of the training tokens. In 367 pairs (2.18%), all words that occur in only one hypothesis are stopwords: without stopwords, model 1 would get the same average vector for h1 and h2 in 357 of them, and model 2 the same input in 263. **Stopwords are kept.**

## Stemming and lemmatization
Stem all lowercased training words with NLTK's Porter stemmer (course slides "NLP Pipeline" (Week 1), slide 15) and measure the GloVe coverage again.

In [13]:
# Porter stemmer, as in the course exercises "NLP Pipeline" and "Topic Modeling"
stemmer = nltk.stem.PorterStemmer()

# the same training tokens, each lowercased word replaced by its stem
fdist_stemmed = nltk.FreqDist()
for word, count in fdist_lower.items():
    fdist_stemmed[stemmer.stem(word)] += count

print('Types (stemmed):', len(fdist_stemmed))
print('GloVe, lowercased and stemmed:')
coverage(wv, fdist_stemmed)

Types (stemmed): 11278
GloVe, lowercased and stemmed:
Known types: 71.05%
Known tokens: 93.42%
Most frequent unknown: ['decid', 'famili', 'becaus', 'realli', 'realiz', 'movi', 'everi', 'excit', 'outsid', 'peopl', 'becam', 'notic', 'practic', 'everyon', 'onli']


The same with NLTK's WordNet lemmatizer (same slides, slide 16), which needs the WordNet resource. A few examples show what it does with verbs and nouns.

In [14]:
# WordNet lemmatizer (needs the WordNet resource)
nltk.download('wordnet')
lemmatizer = nltk.stem.WordNetLemmatizer()

# the same training tokens, each lowercased word replaced by its lemma
fdist_lemmatized = nltk.FreqDist()
for word, count in fdist_lower.items():
    fdist_lemmatized[lemmatizer.lemmatize(word)] += count

print('Types (lemmatized):', len(fdist_lemmatized))
print('GloVe, lowercased and lemmatized:')
coverage(wv, fdist_lemmatized)
# what the lemmatizer does with some verbs and nouns
print([(word, lemmatizer.lemmatize(word)) for word in ['was', 'has', 'went', 'houses', 'flies']])

[nltk_data] Downloading package wordnet to /root/nltk_data...


Types (lemmatized): 14406
GloVe, lowercased and lemmatized:
Known types: 94.87%
Known tokens: 99.82%
Most frequent unknown: ['cooky', 'mturk', '..', 'selfie', 'zaiden', 'a+', 'antonela', 'fariah', 'myah', 'shalil', 'rubix', 'kelsea', 'jacelyn', 'lamay', 'marisse']
[('was', 'wa'), ('has', 'ha'), ('went', 'went'), ('houses', 'house'), ('flies', 'fly')]


Stemming lowers the GloVe coverage of the training tokens from 99.84% to 93.42%: stems like *decid* or *famili* have no vector. Lemmatization keeps it (99.82%) but adds nothing, and inflected forms keep tense and number. Without part-of-speech tags, the WordNet lemmatizer treats every word as a noun and gives wrong lemmas (*was* → *wa*, *has* → *ha*, like *dies* → *dy* on slide 16). **No stemming and no lemmatization are used.**

## Punctuation
Share of the training tokens that are punctuation (tokens that consist only of punctuation characters), and the number of hypothesis pairs whose differing tokens are all punctuation, with what removing the punctuation would do to these pairs, as for the stopwords.

In [15]:
import string

# a token is punctuation if all its characters are punctuation characters (e.g. '.', ',', '``')
def is_punctuation(word):
    return all(character in string.punctuation for character in word)

punctuation_tokens = sum(fdist_lower[word] for word in fdist_lower if is_punctuation(word))
print(f'Punctuation tokens: {punctuation_tokens / fdist_lower.N():.2%}')
print('Punctuation without a GloVe vector:', [word for word in fdist_lower if is_punctuation(word) and word not in wv])

only_punctuation, same_counts, same_sequence = 0, 0, 0
for h1, h2 in zip(train['hypothesis_1'], train['hypothesis_2']):
    # lists of the lowercased words: they keep the word order and repeated words
    words_1 = [word.lower() for word in nltk.word_tokenize(h1)]
    words_2 = [word.lower() for word in nltk.word_tokenize(h2)]
    difference = set(words_1) ^ set(words_2)  # words that occur in only one of the two hypotheses
    if difference and all(is_punctuation(word) for word in difference):
        only_punctuation += 1
        # show the pair
        print(h1, '|', h2)
        # the two hypotheses without their punctuation
        kept_1 = [word for word in words_1 if not is_punctuation(word)]
        kept_2 = [word for word in words_2 if not is_punctuation(word)]
        # same words with the same counts, in any order: model 1 gets the same average vector (True counts as 1)
        same_counts += nltk.FreqDist(kept_1) == nltk.FreqDist(kept_2)
        # same words in the same order: model 2 gets the same input
        same_sequence += kept_1 == kept_2
print('Pairs whose differing tokens are all punctuation:', only_punctuation)
print(f'Share: {only_punctuation / len(train):.2%}')
print('Without punctuation, same words and counts (model 1):', same_counts)
print('Without punctuation, same word sequence (model 2):', same_sequence)

Punctuation tokens: 11.26%
Punctuation without a GloVe vector: ['..']
Albert, decided to spend the day with his friend. | His friend decided to spend the day with Albert.
Samson had backstage passes,. | Backstage passes had Samson.
Stew gave all the caps away. | The caps gave all Stew away.".
Jenny,searched Amazon for a watch for her mother. | Her mother searched Amazon for a watch for Jenny.
Pairs whose differing tokens are all punctuation: 4
Share: 0.02%
Without punctuation, same words and counts (model 1): 4
Without punctuation, same word sequence (model 2): 0


Punctuation is 11.26% of the training tokens, and GloVe has a vector for every punctuation token except `..`. In 4 pairs (0.02%), all differing tokens are punctuation, word swaps with a stray comma or quotation mark (see the output): without punctuation, model 1 would get the same average vector for h1 and h2 in all 4, and model 2 the same input in none. Slide 10 of the course slides "Project Discussions" (Week 2) marks punctuation removal for word2vec, which has no vectors for these tokens; GloVe has them. **Punctuation is kept.**

## Format cleaning
Check all splits for text that needs cleaning (course slides "NLP Pipeline" (Week 1), slide 3: "Remove formatting/HTML?"): HTML tags, web addresses, repeated spaces, and characters outside ASCII.

In [16]:
import re

for name, split in [('train', train), ('valid', valid), ('test', test)]:
    # all texts of the split (O1, O2, h1, h2 of every example)
    texts = [text for column in columns for text in split[column]]
    # HTML tags like <br>, and web addresses
    html = sum(bool(re.search(r'<[^>]+>', text)) for text in texts)
    urls = sum(bool(re.search(r'https?://|www\.', text)) for text in texts)
    # two or more spaces in a row
    spaces = sum('  ' in text for text in texts)
    # texts with characters outside ASCII (code above 127), and these characters with their counts
    non_ascii = sum(any(ord(character) > 127 for character in text) for text in texts)
    characters = nltk.FreqDist(character for text in texts for character in text if ord(character) > 127)
    print(f'{name}: {len(texts)} texts, HTML tags {html}, URLs {urls}, repeated spaces {spaces}, non-ASCII {non_ascii} texts {characters.most_common()}')

# how word_tokenize splits a word with a curly apostrophe
print(nltk.word_tokenize("He didn’t go."))

train: 67204 texts, HTML tags 0, URLs 0, repeated spaces 68, non-ASCII 38 texts [('’', 32), ('�', 8), ('é', 2), ('´', 1)]
valid: 4000 texts, HTML tags 0, URLs 0, repeated spaces 5, non-ASCII 0 texts []
test: 6128 texts, HTML tags 0, URLs 0, repeated spaces 6, non-ASCII 2 texts [('’', 2)]
['He', 'didn', '’', 't', 'go', '.']


No HTML tags or web addresses in any split. Repeated spaces do not matter, because `word_tokenize` splits at any whitespace. Only 38 of the 67,204 training texts (0.06%) and 2 test texts contain characters outside ASCII: mostly the curly apostrophe ’, which `word_tokenize` splits off (*didn’t* → *didn*, *’*, *t*, so the negation *n't* is missing there), and 8 broken characters from encoding errors (*hadn���t*). These cases are rare, so **no format cleaning is done**.

## Preprocessing function
The preprocessing decisions above in one function.

In [17]:
def preprocess(text):
    # tokenize
    text = nltk.word_tokenize(text)
    # lowercase all words
    text = [word.lower() for word in text]
    # remove stopwords (not used, see above)
    # text = [word for word in text if word not in stopwords]
    return text

print(train[269]["hypothesis_1"])
print(preprocess(train[269]["hypothesis_1"]))

He didn't listen to them, and had to go to the hospital.
['he', 'did', "n't", 'listen', 'to', 'them', ',', 'and', 'had', 'to', 'go', 'to', 'the', 'hospital', '.']


# Part 2: input of model 1

## Sentence vectors
Each text becomes one vector: the average of the GloVe vectors of its words (course slides "Word Embeddings", slide 32). The average rather than the other options on slide 21 of the course slides "Project Discussions" (Week 5): the sum grows with the text length, and the first or last token ignores almost all of the text. Words without a GloVe vector are skipped.

In [18]:
words = preprocess(train[269]['hypothesis_1'])
vectors = [wv[word] for word in words if word in wv]
print(len(words), len(vectors))
print(vectors[0].shape)

15 15
(300,)


Average the word vectors: one vector with 300 numbers for the whole sentence.

In [19]:
import numpy as np

sentence_vector = np.mean(vectors, axis=0)
print(sentence_vector.shape)
print(sentence_vector[:5])

(300,)
[-0.21954928  0.03110227 -0.24757503 -0.11346499 -0.11204454]


The same steps as a function that works for any text. A text without any known word in GloVe gets a zero vector.

In [20]:
def average_vector(text):
    words = preprocess(text)
    vectors = [wv[word] for word in words if word in wv]
    # no known word: zero vector
    if len(vectors) == 0:
        return np.zeros(wv.vector_size)
    return np.mean(vectors, axis=0)

print(average_vector(train[269]["hypothesis_1"])[:5])
print(average_vector("mturk").shape)
print(average_vector("mturk")[:5])

[-0.21954928  0.03110227 -0.24757503 -0.11346499 -0.11204454]
(300,)
[0. 0. 0. 0. 0.]


## Input format
One example is one input: the sentence vectors of O1, O2, h1 and h2 side by side (4 x 300 = 1200 numbers). The label (1 or 2) becomes the class 0 or 1.

Order: the columns of the dataset, the observations in their time order, then the hypotheses in the order of the classes (a stated reason for the order: course slides "Project Discussions" (Week 5), slide 27). With fixed positions, another order would only permute the weights of the first layer. Whether the fixed positions matter: `04_model2_rnn.ipynb` (section "Check: does the position of the hypotheses matter?").

Feature selection: all four texts, because the task is to choose the hypothesis that explains the observations. What the observations add: the hypothesis-only run in `03_model1_embeddings.ipynb` and the section "Check: do the observations help?" in `04_model2_rnn.ipynb`.

In [21]:
columns = ["observation_1", "observation_2", "hypothesis_1", "hypothesis_2"]

example = train[0]
x = np.concatenate([average_vector(example[column]) for column in columns])
y = example["label"] - 1
print(x.shape)
print(example["label"], y)

(1200,)
2 1


Apply this to all examples of each split: one row of 1200 numbers per example, and one class per example.

In [22]:
def example_vector(example):
    return np.concatenate([average_vector(example[column]) for column in columns])

X_train = np.array([example_vector(example) for example in train])
y_train = np.array([example["label"] - 1 for example in train])
print(X_train.shape, y_train.shape)

(16801, 1200) (16801,)


In [23]:
X_valid = np.array([example_vector(example) for example in valid])
y_valid = np.array([example["label"] - 1 for example in valid])
X_test = np.array([example_vector(example) for example in test])
y_test = np.array([example["label"] - 1 for example in test])
print(X_valid.shape, y_valid.shape)
print(X_test.shape, y_test.shape)

(1000, 1200) (1000,)
(1532, 1200) (1532,)


Save the features and labels of all splits to one file, so that the training notebooks can load them without GloVe.

In [24]:
import os

os.makedirs("features", exist_ok=True)
np.savez("features/model1.npz",
         X_train=X_train.astype(np.float32), y_train=y_train,
         X_valid=X_valid.astype(np.float32), y_valid=y_valid,
         X_test=X_test.astype(np.float32), y_test=y_test)
print(os.path.getsize("features/model1.npz") / 1e6, "MB")

92.954558 MB


# Part 3: input of model 2

## Model 2: text lengths in tokens
Measure the length of the training texts in tokens after preprocessing, to choose the padding length for model 2.

In [25]:
# Model 2 reads every text word by word, so the length in tokens matters
for column in columns:
    # number of tokens of each training text in this column (same preprocess() as model 1)
    lengths = [len(preprocess(text)) for text in train[column]]
    # average length, longest text, and the length that 99% of the texts do not exceed
    print(f'{column}: mean {np.mean(lengths):.1f}, max {max(lengths)}, 99% <= {np.percentile(lengths, 99):.0f}')

observation_1: mean 9.2, max 20, 99% <= 15
observation_2: mean 10.6, max 20, 99% <= 17
hypothesis_1: mean 9.6, max 34, 99% <= 20
hypothesis_2: mean 9.6, max 33, 99% <= 21


## Model 2: padding
All texts are padded to the longest training text, 34 tokens, so no training word is lost. Longer texts are cut: only both hypotheses of `valid[490]` (35 tokens) lose their final period, and no test text is that long (longest: 28 tokens, see the check below). The padding (index 0) goes at the end of each text (course slides "Project Discussions" (Week 4), slide 13). `pack_padded_sequence` (same slides, slide 17) gives the RNN the real lengths, so h_n is the state after the last real word.

In [26]:
# all texts get the length of the longest training text (see the output above)
max_length = 34

def pad(numbers):
    # cut texts that are longer than max_length (only happens in the validation set, see above)
    numbers = numbers[:max_length]
    # zeros (padding) at the end, so that the real words come first
    return numbers + [0] * (max_length - len(numbers))

# test with made-up token indices (the real ones come from the vocabulary, next step)
padded = pad([14, 102, 57, 1893, 3])
print(padded)
print(len(padded))


[14, 102, 57, 1893, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
34


## Model 2: vocabulary
Every lowercased word of the three splits that has a GloVe vector: first the training words by frequency (the most frequent gets index 1), then the other validation and test words in the order they appear. So model 2 skips the same words as model 1. The other words only add rows to the frozen embedding matrix: the training input does not change, and no labels are used (own vocabulary vs. pretrained: course slides "Project Discussions" (Week 5), slide 20). Index 0 is the padding. Rare words are kept, because their vectors come from GloVe's much larger corpus. Words without a vector (0.16% of the training tokens, mostly typos, hyphenated compounds and rare names) are skipped instead of mapped to an UNK token (same slides (Week 2), slide 10): with frozen embeddings, an UNK vector could not learn.

In [27]:
# vocabulary: word -> index, for every training word that has a GloVe vector
vocab = {}
# most_common(): all training words, the most frequent first
for word, count in fdist_lower.most_common():
    # words without a GloVe vector are skipped (as in model 1)
    if word in wv:
        # next free index: 1 for the first word, 2 for the second, ... (0 stays free for padding)
        vocab[word] = len(vocab) + 1

# then the other validation and test words that have a GloVe vector (as in model 1); the training words keep their indices
for split in [valid, test]:
    for column in columns:
        for text in split[column]:
            for word in preprocess(text):
                if word in wv and word not in vocab:
                    vocab[word] = len(vocab) + 1

print('Vocabulary size:', len(vocab))
# the first 5 entries (word, index)
print(list(vocab.items())[:5])
# look up the index of one word
print(vocab['hospital'])

Vocabulary size: 16454
[('.', 1), ('the', 2), ('to', 3), ('a', 4), ('was', 5)]
527


## Model 2: text to token indices
Each text is preprocessed as for model 1, every word is replaced by its index in the vocabulary (words that are not in the vocabulary are skipped), and the list is padded to 34 token indices.


In [28]:
def to_numbers(text):
    # same preprocessing as model 1: tokenize, lowercase
    words = preprocess(text)
    # the index of each word; words that are not in the vocabulary are skipped
    numbers = [vocab[word] for word in words if word in vocab]
    # zeros at the end, up to 34 token indices
    return pad(numbers)

# test with the example sentence from the tokenization section
text = train[269]['hypothesis_1']
print(text)
print(to_numbers(text))


He didn't listen to them, and had to go to the hospital.
[8, 28, 23, 1296, 3, 73, 13, 6, 17, 3, 47, 3, 2, 527, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


## Model 2: embedding matrix
Row i is the GloVe vector of the word with index i, and row 0 (padding) stays all zeros. The embedding layer of model 2 uses this matrix, frozen as in model 1.

In [29]:
# one row per index (0 = padding, 1 ... len(vocab) = words), 300 columns (the GloVe vector size)
embedding_matrix = np.zeros((len(vocab) + 1, wv.vector_size), dtype=np.float32)
# fill row "number" with the GloVe vector of the word; row 0 stays all zeros
for word, number in vocab.items():
    embedding_matrix[number] = wv[word]

print(embedding_matrix.shape)
# row 0 (padding): first 5 numbers
print(embedding_matrix[0][:5])
# row of "hospital" and the GloVe vector of "hospital": should be the same
print(embedding_matrix[vocab['hospital']][:5])
print(wv['hospital'][:5])


(16455, 300)
[0. 0. 0. 0. 0.]
[-0.87315   -0.62477   -0.096836   0.0025302 -0.010344 ]
[-0.87315   -0.62477   -0.096836   0.0025302 -0.010344 ]


## Model 2: input format
One example becomes 4 rows of 34 token indices: O1, O2, h1 and h2, in the same order as for model 1. Apply this to all examples of each split (one array of shape [examples, 4, 34] per split). They are stored as integers (`int64`), because the embedding layer needs integer indices. The labels are the same as for model 1.

In [30]:
# one example: 4 rows of 34 token indices (O1, O2, h1, h2, in the order of `columns`)
def example_numbers(example):
    return [to_numbers(example[column]) for column in columns]

# all examples of each split, shape (examples, 4, 34)
# X2 = input of model 2 (X_train, X_valid, X_test above are the input of model 1)
X2_train = np.array([example_numbers(example) for example in train], dtype=np.int64)
X2_valid = np.array([example_numbers(example) for example in valid], dtype=np.int64)
X2_test = np.array([example_numbers(example) for example in test], dtype=np.int64)
print(X2_train.shape, X2_valid.shape, X2_test.shape)
# check: h1 of train[269] (row 2) is the same as in the test of to_numbers() above
print(X2_train[269][2])

(16801, 4, 34) (1000, 4, 34) (1532, 4, 34)
[   8   28   23 1296    3   73   13    6   17    3   47    3    2  527
    1    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0]


Length of each text in token indices (all indices that are not 0). `pack_padded_sequence` in model 2 needs a length of at least 1 for every text, so check the shortest text of each split.

In [31]:
for name, X2 in [('train', X2_train), ('valid', X2_valid), ('test', X2_test)]:
    # number of real token indices (not 0 = padding) in every text, shape (examples, 4)
    lengths = (X2 != 0).sum(axis=-1)
    print(f'{name}: shortest text {lengths.min()}, longest text {lengths.max()} token indices')

train: shortest text 1, longest text 34 token indices
valid: shortest text 3, longest text 34 token indices
test: shortest text 2, longest text 28 token indices


Share of the tokens of each split that are in the vocabulary. Words that are not in it are skipped: in every split only words without a GloVe vector.

In [32]:
for name, split in [('train', train), ('valid', valid), ('test', test)]:
    # all tokens of the split (O1, O2, h1 and h2 of every example)
    words = [word for column in columns for text in split[column] for word in preprocess(text)]
    # count the tokens that have an index in the vocabulary (True counts as 1)
    known = sum(word in vocab for word in words)
    print(f'{name}: {known / len(words):.2%} of the tokens are in the vocabulary')

train: 99.84% of the tokens are in the vocabulary
valid: 99.89% of the tokens are in the vocabulary
test: 99.84% of the tokens are in the vocabulary


Almost all tokens are kept: 99.84% of the training tokens and 99.89% / 99.84% of the validation / test tokens. Only words without a GloVe vector are skipped, the same words as in model 1.

Save the token indices, the labels and the embedding matrix of all splits to one file, so that the training notebook of model 2 can load them without GloVe (as for model 1).

In [34]:
# same format as for model 1; the embedding matrix is saved too, for the embedding layer of model 2
np.savez("features/model2.npz",
         X_train=X2_train, y_train=y_train,
         X_valid=X2_valid, y_valid=y_valid,
         X_test=X2_test, y_test=y_test,
         embedding_matrix=embedding_matrix)
print(os.path.getsize("features/model2.npz") / 1e6, "MB")

40.936726 MB


## Sources
Course materials: NLP HS26 by A. Marfurt, Lucerne University of Applied Sciences and Arts (HSLU), 2026. Full references in README.md.
- Dataset: course version of ART, `amarfurt/art-nlp-hs26` on Hugging Face (A. Marfurt), based on Bhagavatula et al. (2020), *Abductive Commonsense Reasoning*, ICLR 2020, https://arxiv.org/abs/1908.05739.
- Loading the splits: code from slide 9 of the course slides "Course Project HS26" (A. Marfurt).
- Tokenization: `nltk.download('punkt_tab')` and `nltk.word_tokenize` as in the course exercise "NLP Pipeline" (Week 1).
- Tokens/types and `nltk.FreqDist` as in the course exercise "NLP Pipeline" (Week 1); coverage check (`word in wv`).
- Word embeddings: GloVe (Pennington et al., 2014), fastText (Bojanowski et al., 2017) and word2vec (Mikolov et al., 2013), loaded with gensim's downloader (Řehůřek & Sojka, 2010) as in the course exercise "Word Embeddings" (Week 2).
- Stopwords: NLTK's English stopword list (`nltk.corpus.stopwords`) as in the course exercise "Topic Modeling" (Week 2); set operations `^` and `<=`, Python documentation, https://docs.python.org/3/builtins/stdtypes.html#set-types-set-frozenset; comparing lists with `==`, Python tutorial "Comparing Sequences and Other Types", https://docs.python.org/3/tutorial/datastructures.html#comparing-sequences-and-other-types; comparing word counts with `nltk.FreqDist`, NLTK documentation, https://www.nltk.org/api/nltk.probability.FreqDist.html.
- Stemming and lemmatization: `nltk.stem.PorterStemmer` and `nltk.stem.WordNetLemmatizer` from the course slides "NLP Pipeline" (Week 1), slides 15 and 16 (the Porter stemmer also in the course exercises "NLP Pipeline" (Week 1) and "Topic Modeling" (Week 2)); `nltk.download('wordnet')`, NLTK documentation "Installing NLTK Data", https://www.nltk.org/data.html; adding counts to an `nltk.FreqDist`, NLTK documentation, https://www.nltk.org/api/nltk.probability.FreqDist.html.
- Punctuation: `string.punctuation`, Python documentation, https://docs.python.org/3/library/string.html#string.punctuation; built-in function `all()`, https://docs.python.org/3/library/functions.html#all; pairs whose differing tokens are all punctuation counted as for the stopwords; punctuation removal for word2vec from the course slides "Project Discussions" (Week 2), slide 10.
- Format cleaning: "Remove formatting/HTML?" from the course slides "NLP Pipeline" (Week 1), slide 3, and cleaning HTML/emojis as a preprocessing decision from the course slides "Project Discussions" (Week 2), slide 10; `re.search`, Python documentation, https://docs.python.org/3/library/re.html; built-in functions `any()` and `ord()`, https://docs.python.org/3/library/functions.html.
- Preprocessing function: structure of `preprocess()` as in the course exercise "Topic Modeling" (Week 2).
- Sentence vectors: averaging the word vectors of a text as input for a classifier, from the course slides "Word Embeddings" (Week 2), slide 32 "Extrinsic Evaluation"; average, sum, first or last token as input to the classifier from the course slides "Project Discussions" (Week 5), slide 21; word vector lookup (`wv[word]`) as in the course exercise "Word Embeddings" (Week 2); `np.mean`, `np.zeros`, `np.concatenate` and `np.array` from NumPy (Harris et al., 2020), https://numpy.org/doc/stable/.
- Naming: `X` (features, uppercase because it is a matrix) and `y` (labels) following the scikit-learn convention, https://scikit-learn.org/stable/glossary.html.
- Saving the features: `np.savez` and `ndarray.astype` as in the NumPy documentation, https://numpy.org/doc/stable/reference/generated/numpy.savez.html and https://numpy.org/doc/stable/reference/generated/numpy.ndarray.astype.html; `os.makedirs` and `os.path.getsize` from the Python standard library, https://docs.python.org/3/library/os.html.
- Text lengths: `np.percentile` from NumPy (Harris et al., 2020), https://numpy.org/doc/stable/reference/generated/numpy.percentile.html.
- Padding: a batch has one sequence length, so texts of different lengths are filled up (course slides "Preliminaries" (Week 1), slides 14-17); padding at the end of the text, pad tokens ignored in the computation, and `pack_padded_sequence` for padding in RNNs from the course slides "Project Discussions" (Week 4, "Data Loading"), slides 13 and 17, and the PyTorch documentation `torch.nn.utils.rnn.pack_padded_sequence`, https://docs.pytorch.org/docs/stable/generated/torch.nn.utils.rnn.pack_padded_sequence.html; list slicing, `*` and `+` from the Python documentation "Common Sequence Operations", https://docs.python.org/3/library/stdtypes.html#common-sequence-operations; padding and truncation to a fixed length as with `padding="max_length", truncation=True` of the tokenizer in the course exercise "Experiment Tracking".
- Vocabulary: each word gets its position in the vocabulary, as for one-hot vectors in the course slides "Vector Space Model" (Week 2), slide 5; built from the training words with `nltk.FreqDist` and `most_common()` as in the course exercise "NLP Pipeline" (Week 1), then the other validation and test words; own vocabulary vs. pretrained from the course slides "Project Discussions" (Week 5), slide 20; `word in wv` as in the course exercise "Word Embeddings" (Week 2); Python dictionaries, https://docs.python.org/3/tutorial/datastructures.html#dictionaries.
- Text to token indices: list comprehension with an `if` condition, Python tutorial "List Comprehensions", https://docs.python.org/3/tutorial/datastructures.html#list-comprehensions.
- Embedding matrix: input embedding matrix of size vocabulary size x vector size from the course slides "Preliminaries" (Week 1), slide 18; looking up a row is the same as multiplying a one-hot vector with the matrix, as in the word2vec architecture (course slides "Word Embeddings" (Week 2), slide 8); pretrained word vectors as the embedding matrix of a neural network, course slides "Word Embeddings" (Week 2), slide 23; `np.zeros` with `shape` and `dtype` from the NumPy documentation, https://numpy.org/doc/stable/reference/generated/numpy.zeros.html; `dict.items()` from the Python tutorial "Looping Techniques", https://docs.python.org/3/tutorial/datastructures.html#looping-techniques; word vector lookup (`wv[word]`) as in the course exercise "Word Embeddings" (Week 2); row 0 for padding as `padding_idx` in the PyTorch documentation `torch.nn.Embedding`, https://docs.pytorch.org/docs/stable/generated/torch.nn.Embedding.html.
- Input of model 2: arrays of token indices with `np.array(..., dtype=np.int64)`, because the embedding layer needs indices (PyTorch documentation `torch.nn.Embedding`: input "IntTensor or LongTensor ... containing the indices to extract"); nested list comprehension from the Python tutorial "List Comprehensions"; counting with `sum()` over True/False values; saving with `np.savez` as for model 1 (see above). Text lengths for `pack_padded_sequence`: counting the non-zero indices with `(X2 != 0).sum(axis=-1)` (NumPy documentation `numpy.sum`, https://numpy.org/doc/stable/reference/generated/numpy.sum.html).
- Unknown words: handling unknown and rare words ("map to UNK") as a preprocessing decision from the course slides "Project Discussions" (Week 2), slide 10.
- Small functions: `str.split`, `str.lower`, `set` and `list.extend`, Python documentation, https://docs.python.org/3/library/stdtypes.html; `zip()` and `enumerate()`, https://docs.python.org/3/library/functions.html; `FreqDist.N()`, NLTK documentation, https://www.nltk.org/api/nltk.probability.FreqDist.html; the default part of speech of `lemmatize` (noun), https://www.nltk.org/api/nltk.stem.wordnet.html; `ndarray.min` and `ndarray.max`, NumPy documentation, https://numpy.org/doc/stable/reference/generated/numpy.ndarray.min.html and https://numpy.org/doc/stable/reference/generated/numpy.ndarray.max.html.
- Tools: Claude (Anthropic), ChatGPT (OpenAI) and Gemini (Google), see "Use of AI" in README.md; all code reviewed and understood by me.
- Input format: the texts side by side as classifier input, the same form as in the previous semester's example in the course slides "Project Discussions" (Week 5), slide 26; a stated reason for the order of the concatenated inputs from the course slides "Project Discussions" (Week 5), slide 27; feature selection as a decision from the course document "Grading criteria HS26" (A. Marfurt).
- Setup on Colab: `drive.mount`, Google Colab, https://colab.research.google.com/; `sys.modules`, https://docs.python.org/3/library/sys.html#sys.modules, and `os.chdir`, https://docs.python.org/3/library/os.html#os.chdir, Python documentation; `%pip`, IPython documentation, https://ipython.readthedocs.io/en/stable/interactive/magics.html.